In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Let's see a single Head perform self-attention under the hood 

In [ ]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32 # batch, time (context), channels (embedding size)
# input tensor after embedding the input tokens
X = torch.randn(B, T, C)

n_heads = 4
head_size = C // n_heads
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
# all of them are comming from the same input X, so they are correlated
k = key(X)   # (B, T, head_size)
q = query(X) # (B, T, head_size)
v = value(X) # (B, T, head_size)

weights = q @ k.transpose(1, 2) * head_size**-0.5 # (B, T, T)
# mask out the lower triangle (causal attention)
weights = weights.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
weights = F.softmax(weights, dim=-1) # (B, T, T)
out = weights @ v # (B, T, head_size)
# simulate multiple heads of self-attention in parallel
out1 = torch.randn_like(out)
out2 = torch.randn_like(out)
out3 = torch.randn_like(out)
# concatenate the outputs of the heads
out4 = torch.cat([out, out1, out2, out3], dim=-1) # (B, T, head_size * num_heads)
out4 = nn.Linear(head_size * n_heads, C, bias=False)(out4) # (B, T, C)
out4.shape

torch.Size([4, 8, 32])

Let's implement this in classes

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
    def forward(self, x):
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)

        weight = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, T)
        weight = weight.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
        weight = F.softmax(weight, dim=-1) # (B, T, T)

        out = weight @ v # (B, T, head_size)
        return out
    
class MultiHeadAttention(nn.Module):
    """Multible heads of self-attention in parallel"""
    
    def __init__(self, n_embd, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd) 
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

torch.Size([4, 8, 5])